# 📚 Etapa 1 — Geração de Dataset para Fine-Tuning (RAG)

Este notebook cria um dataset no formato `Instruction` / `Output` a partir do documento PDF atribuído ao aluno (`acupuntura.pdf`).

**Modelo gerador:** `google/gemma-2-2b-it` — 2.6 bilhões de parâmetros (> 1.5B conforme exigido).

**Estratégia de chunking:** Texto dividido em blocos de até **500 caracteres** (padrão) e **800 caracteres** (alternativa), com comparação dos pares gerados.

**Saída:** arquivo `dataset_gerado.jsonl` com ao menos 100 pares válidos.

---
## Etapas:
1. Instalação das dependências
2. Extração de texto do PDF
3. Divisão em chunks (500 e 800 chars)
4. Geração de pares Instrução/Resposta via Gemma 2 2B Instruct
5. Curadoria e salvamento em JSONL


## 1. Instalação das bibliotecas

- `transformers`, `torch`, `accelerate`: para carregar e executar o Gemma 2 2B Instruct.
- `pdfplumber`: extração de texto de PDFs.
- `tqdm`: barra de progresso.
- `bitsandbytes`: quantização opcional para rodar o Gemma em CPU/GPU com menos memória.


In [13]:
%pip install transformers huggingface_hub torch accelerate pdfplumber tqdm bitsandbytes

Note: you may need to restart the kernel to use updated packages.


## 2. Importações e configurações

Importamos os módulos e suprimimos avisos desnecessários.


In [6]:
import pdfplumber
import json
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, logging, BitsAndBytesConfig
from tqdm import tqdm

logging.set_verbosity_error()
print(f'PyTorch: {torch.__version__}')
print(f'Dispositivo: {"GPU" if torch.cuda.is_available() else "CPU"}')


PyTorch: 2.12.0+cu130
Dispositivo: CPU


## 3. Extração de texto do PDF

Utilizamos `pdfplumber` para extrair o texto do documento `acupuntura.pdf`, que é o documento base deste projeto.


In [7]:
def extract_text_from_file(file_path):
    """
    Extrai texto de um arquivo .pdf ou .txt.
    Retorna uma string com todo o conteúdo.
    """
    if file_path.lower().endswith('.pdf'):
        text = ""
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        return text
    elif file_path.lower().endswith('.txt'):
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    else:
        raise ValueError("Formato não suportado. Use .pdf ou .txt")

# Extrai o texto do documento atribuído
ARQUIVO_PDF = "acupuntura.pdf"
texto_completo = extract_text_from_file(ARQUIVO_PDF)
print(f'✅ Texto extraído: {len(texto_completo):,} caracteres')
print(f'Prévia:\n{texto_completo[:300]}')


✅ Texto extraído: 360,495 caracteres
Prévia:
ACUPUNTURA
CLÁSSICA CHINESA
Tom Sintan Wen
DR. TOM SINTAN WEN
ACUPUNTURA
CLÁSSICA
CHINESA
EDITORA CULTRIX
Copyright (c) 1985 Dr. Tom Sintan Wen
Direitos reservados
EDITORA CULTRIX LTDA.
Rua Dr. Mário Vicente, 374 - 04270 São Paulo, SP - Fone: 63 3141
SUMÁRIO
Prefácio, 7
I. Introdução, 9
II. Ás Teori


## 4. Divisão em chunks (chunking)

### Justificativa do max_chunk_length:
- **500 caracteres** (padrão): chunks focados, menor risco de ultrapassar o limite de tokens do modelo. Ideal para gerar perguntas objetivas e diretas.
- **800 caracteres** (alternativa): mais contexto por chunk, mas com maior risco de truncamento nos modelos menores.

Testamos ambos abaixo para comparar a qualidade dos pares gerados.


In [8]:
def split_text(text, max_chunk_length=500):
    """
    Divide o texto em chunks com base em quebras de parágrafo.
    Cada chunk terá no máximo max_chunk_length caracteres.
    """
    paragraphs = text.split("\n")
    chunks = []
    current_chunk = ""

    for para in paragraphs:
        if len(current_chunk) + len(para) < max_chunk_length:
            current_chunk += para + "\n"
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = para + "\n"

    if current_chunk:
        chunks.append(current_chunk.strip())

    return [c for c in chunks if len(c.strip()) >= 20]

# Comparação dos dois tamanhos de chunk
chunks_500 = split_text(texto_completo, max_chunk_length=500)
chunks_800 = split_text(texto_completo, max_chunk_length=800)

print(f'Chunks com 500 chars: {len(chunks_500)}')
print(f'Chunks com 800 chars: {len(chunks_800)}')
print(f'\nExemplo chunk (500):\n{chunks_500[0]}')


Chunks com 500 chars: 770
Chunks com 800 chars: 469

Exemplo chunk (500):
ACUPUNTURA
CLÁSSICA CHINESA
Tom Sintan Wen
DR. TOM SINTAN WEN
ACUPUNTURA
CLÁSSICA
CHINESA
EDITORA CULTRIX
Copyright (c) 1985 Dr. Tom Sintan Wen
Direitos reservados
EDITORA CULTRIX LTDA.
Rua Dr. Mário Vicente, 374 - 04270 São Paulo, SP - Fone: 63 3141
SUMÁRIO
Prefácio, 7
I. Introdução, 9
II. Ás Teorias Básicas da Medicina Chinesa, 18
Teoria do Yin-Yang, 18
Teoria dos cinco elementos, 21
Teoria dos meridianos, 25
III.Os Pontos de Diagnóstico e os Tratamentos da Medicina Chinesa, 30


## 5. Carregamento do Gemma 2 2B Instruct

O modelo `google/gemma-2-2b-it` é um modelo instrução-tunado com 2.6 bilhões de parâmetros, da família Gemma 2 do Google DeepMind. Ele supera modelos maiores em benchmarks como MMLU e HellaSwag, sendo ideal para geração de pares QA em português e inglês.

> **Nota:** Requer aceitar os Termos de Uso no HuggingFace e autenticação via `HF_TOKEN`.


## 5a. Autenticação no HuggingFace

O modelo `google/gemma-2-2b-it` é **gated** (restrito). Para acessá-lo é necessário:
1. Criar uma conta em [huggingface.co](https://huggingface.co)
2. Aceitar os termos do Gemma em: https://huggingface.co/google/gemma-2-2b-it
3. Gerar um token de acesso em: https://huggingface.co/settings/tokens
4. Cole seu token na célula abaixo (ou defina a variável de ambiente `HF_TOKEN`).


In [10]:
import os
from huggingface_hub import login

# Opção 1: Token via variável de ambiente (recomendado para segurança)
# hf_token = os.environ.get("hf_********************************", None)

# Opção 2: Cole seu token aqui (NÃO commite com o token exposto no Git!)
hf_token = "hf_********************************"

if hf_token:
    login(token=hf_token)
    print('✅ Autenticado no HuggingFace com token da variável HF_TOKEN.')
else:
    # Login interativo — abre prompt para colar o token
    print('⚠️  HF_TOKEN não encontrado. Abrindo login interativo...')
    login()


✅ Autenticado no HuggingFace com token da variável HF_TOKEN.


In [12]:
MODEL_ID = "google/gemma-2-2b-it"

print(f'🔄 Carregando modelo: {MODEL_ID} ...')

# Pipeline para geração de texto (chat/instruction-following)
hf_pipeline = pipeline(
    "text-generation",
    model=MODEL_ID,
    device_map="auto",
    dtype=torch.bfloat16,
    trust_remote_code=True
)

print('✅ Modelo carregado com sucesso!')


🔄 Carregando modelo: google/gemma-2-2b-it ...


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-2-2b-it.
403 Client Error. (Request ID: Root=1-6a2b0356-1a490afc2f2e69b06bf03d85;3b8d290b-b45a-4bf7-8f64-a321c8f7f0ed)

Cannot access gated repo for url https://huggingface.co/google/gemma-2-2b-it/resolve/main/config.json.
Access to model google/gemma-2-2b-it is restricted and you are not in the authorized list. Visit https://huggingface.co/google/gemma-2-2b-it to ask for access.

## 6. Geração de Pares (Instrução, Resposta)

O modelo recebe um prompt estruturado pedindo que gere uma pergunta objetiva (< 15 palavras) e uma resposta direta (< 20 palavras) com base no chunk fornecido.

**Formato obrigatório de saída:**
```json
{"Instruction": "...", "Output": "..."}
```


In [ ]:
def generate_instruction_response(chunk, hf_pipeline):
    """
    Usa o modelo para gerar um par de QA estruturado focado em Acupuntura Clássica Chinesa.
    """
    prompt = f"""Você é um especialista e professor de Acupuntura Clássica Chinesa.
Baseado exclusivamente no trecho de texto abaixo, elabore uma pergunta e uma resposta clínica ou teórica (sobre meridianos, pontos, conceitos de Yin/Yang, diagnóstico, técnicas, etc).

Regras obrigatórias:
1. A pergunta deve ser objetiva (máximo 15 palavras).
2. A resposta deve ser direta e factual baseada no texto (máximo 25 palavras).
3. Responda APENAS com um JSON puro, sem blocos de código markdown ou explicações.

Formato esperado:
{{"Instruction": "Sua pergunta de acupuntura aqui", "Output": "Sua resposta curta aqui"}}

Trecho de texto:
{chunk}
"""
    
    messages = [{"role": "user", "content": prompt}]
    
    input_text = hf_pipeline.tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    outputs = hf_pipeline(
        input_text,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.3,
        return_full_text=False
    )
    
    resposta_bruta = outputs[0]["generated_text"].strip()
    
    import re
    import json
    try:
        match = re.search(r"\{.*\}", resposta_bruta, re.DOTALL)
        if match:
            parsed = json.loads(match.group(0))
        else:
            parsed = json.loads(resposta_bruta)
        
        return parsed.get("Instruction"), parsed.get("Output")
    except Exception:
        return None, None


## 7. Salvamento em JSONL

Cada linha do arquivo seguirá o formato obrigatório:
```json
{"Instruction": "...", "Output": "..."}
```


In [ ]:
def save_to_jsonl(pairs, output_file="dataset_gerado.jsonl"):
    """
    Salva pares (instrução, resposta) em formato JSONL.
    """
    with open(output_file, "w", encoding="utf-8") as f:
        for instruction, answer in pairs:
            if instruction and answer:
                example = {
                    "Instruction": instruction,
                    "Output": answer
                }
                f.write(json.dumps(example, ensure_ascii=False) + "\n")
    print(f'💾 Arquivo salvo: {output_file} ({len(pairs)} pares)')


## 8. Geração Completa do Dataset

A função principal coordena todo o fluxo:
- Carrega o modelo Gemma 2 2B
- Processa os chunks do `acupuntura.pdf`
- Salva os pares válidos em `dataset_gerado.jsonl`

> **Dica:** Para teste inicial, use `max_chunks=10`. Para o dataset completo (≥ 100 pares), remova o limite.


In [ ]:
def generate_dataset(chunks, output_file="dataset_gerado.jsonl"):
    """
    Processa os chunks e gera o dataset JSONL.
    """
    print(f'🧠 Gerando pares para {len(chunks)} chunks...')
    pairs = []

    for chunk in tqdm(chunks, desc="Processando chunks"):
        if len(chunk.strip()) < 20:
            continue
        instruction, answer = generate_instruction_response(chunk, hf_pipeline)
        if instruction and answer:
            pairs.append((instruction, answer))

    save_to_jsonl(pairs, output_file)
    print(f'\n✅ Dataset gerado: {len(pairs)} pares válidos')
    return pairs


## 9. Execução — Chunk de 500 chars (padrão)

Processa o documento completo com `max_chunk_length=500`. Para o dataset mínimo de 100 pares, certifique-se de que o documento tem chunks suficientes.


In [ ]:
# Execução com chunks de 500 caracteres (padrão da diretriz)
print(f'Total de chunks (500 chars): {len(chunks_500)}')

pairs_500 = generate_dataset(
    chunks=chunks_500,
    output_file="dataset_gerado.jsonl"
)

print(f'\n📊 Relatório de Curadoria (chunk 500):')
print(f'  Chunks processados: {len(chunks_500)}')
print(f'  Pares gerados: {len(pairs_500)}')
print(f'  Taxa de sucesso: {len(pairs_500)/len(chunks_500)*100:.1f}%')


## 10. Comparação — Chunk de 800 chars (alternativa)

Chunks maiores fornecem mais contexto ao modelo, mas com risco de truncamento e menor foco da pergunta gerada.


In [ ]:
# Execução com chunks de 800 caracteres para comparação
# Usamos max_chunks para não duplicar o dataset principal
print(f'Total de chunks (800 chars): {len(chunks_800)}')

pairs_800_sample = []
for chunk in tqdm(chunks_800[:20], desc="Chunks 800 (amostra)"):
    instr, ans = generate_instruction_response(chunk, hf_pipeline)
    if instr and ans:
        pairs_800_sample.append((instr, ans))

print(f'\n📊 Comparação:')
print(f'  Chunk 500 — {len(pairs_500)} pares de {len(chunks_500)} chunks')
print(f'  Chunk 800 — {len(pairs_800_sample)} pares de 20 chunks (amostra)')
print(f'\n💡 Decisão: chunk_length=500 selecionado por melhor foco e menor risco de truncamento.')


## 11. Verificação Final do Dataset

Confirma que o `dataset_gerado.jsonl` foi salvo corretamente e exibe amostras.


In [ ]:
# Lê e exibe as primeiras amostras do dataset gerado
samples_check = []
with open('dataset_gerado.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            samples_check.append(json.loads(line))

print(f'✅ Dataset final: {len(samples_check)} pares')
print(f'Requisito mínimo (100 pares): {"✅ OK" if len(samples_check) >= 100 else "❌ Insuficiente - gere mais chunks"}')
print(f'\nPrimeiras 3 amostras:')
for i, s in enumerate(samples_check[:3]):
    print(f'\n[{i+1}] Instruction: {s["Instruction"]}')
    print(f'    Output: {s["Output"]}')
